In [16]:
from typing import TypedDict
from  langgraph.graph import StateGraph , START, END

In [17]:
# Define the state
class BatsmanState(TypedDict):
    runs: int
    balls : int
    fours : int
    sixes : int
    sr : float
    bpb : float
    boundary_percent : float
    summary : str

In [18]:
# Define the methord
def calculate_sr(state : BatsmanState) -> BatsmanState:
    # define the properties
    runs = state['runs']
    balls = state['balls']

    sr = (runs/balls)*100
    return {'sr': sr}

def calulate_bpb(state : BatsmanState) -> BatsmanState:
    # define the properties
    balls =  state['balls']
    fours = state['fours']
    sixes = state['sixes']

    boundaries = fours + sixes
    return {
    "bpb": 0 if boundaries == 0 else balls / boundaries
}

def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    # define the properties
    fours = state['fours']
    sixes = state['sixes']
    runs = state['runs']

    boundary_runs = (fours * 4) + (sixes * 6)
    return {"boundary_percent" :
        0 if runs == 0 else (boundary_runs / runs) * 100
    }
    

def summary(state : BatsmanState) -> BatsmanState:
    summary = f"""
Strike Rate - {state['sr']}\n 
Balls per boundary - {state['bpb']}\n
Boundary percent - {state['boundary_percent']}
"""
    return {
    "summary": summary
}


In [19]:
graph = StateGraph(BatsmanState)

# add nodes
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calulate_bpb)
graph.add_node('calculate_boundary_percent',calculate_boundary_percent)
graph.add_node('summary',summary)

# add Edges
graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percent')
graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percent','summary')
graph.add_edge('summary',END)

workflow = graph.compile()
intial_state = {
    'runs': 100,
    'balls' : 50,
    'fours' : 6,
    'sixes': 4
}

result = workflow.invoke(intial_state)
print(result)


{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'sr': 200.0, 'bpb': 5.0, 'boundary_percent': 48.0, 'summary': '\nStrike Rate - 200.0\n \nBalls per boundary - 5.0\n\nBoundary percent - 48.0\n'}
